# DineIQ — Restaurant Intelligence & Analytics Engine
## End-to-End Big Data, Machine Learning, and Econometric Simulation Pipeline

---

### Project Overview
**DineIQ** is an enterprise-grade restaurant analytics platform designed to solve core operational, menu, customer, and pricing challenges facing multi-location food & beverage enterprises.

This notebook demonstrates the end-to-end analytical workflow:
1. **Data Ingestion & Hygiene Assessment**: Audit of raw multi-table transaction and operations data.
2. **Exploratory Data Analysis (EDA)**: Revenue trends, channel performance, and order dynamics.
3. **Multi-Indicator Menu Classification**: BCG quadrant classification (Stars, Plowhorses, Puzzles, Dogs) integrating volume, contribution margin, kitchen wastage, repeat purchase rates, and rating metrics.
4. **Customer Intelligence & RFM Segmentation**: Quantile-based RFM scoring and lifecycle risk tracking.
5. **Market Basket Affinity Analysis**: Co-occurrence mining and association rules (Support, Confidence, Lift) for cross-selling.
6. **Chronological Demand Forecasting**: Time-series feature engineering with strict leak-free splits.
7. **Machine Learning Model Lab**: Multi-class predictive modeling comparing Random Forest, Logistic Regression, and XGBoost.
8. **Dual-Pipeline Cross-Engine Verification**: Independent model inference validation and disagreement audit.
9. **Prescriptive Action Engine & What-If Econometric Simulation**: Price elasticity and scenario modeling.


### 1. Environment Setup & Library Imports
Initialize core libraries for data processing, statistical visualization, machine learning, and metric evaluation.


In [ ]:
import os
import sys
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import xgboost as xgb

# Visual formatting
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110

print("DineIQ Analytics Engine environment ready.")


### 2. Data Ingestion & Quality Audit
Locate and ingest the primary operational datasets across tables:
- `orders.csv`
- `order_items.csv`
- `menu_items.csv`
- `menu_categories.csv`
- `restaurants.csv`
- `customers.csv`
- `wastage.csv`
- `ratings.csv`


In [ ]:
# Determine base data directory
base_dir = Path.cwd()
if (base_dir / "sample_data").exists():
    data_dir = base_dir / "sample_data"
elif (base_dir.parent / "sample_data").exists():
    data_dir = base_dir.parent / "sample_data"
else:
    data_dir = base_dir / "processed_data" / "cleaned_csv"

print(f"Loading operational datasets from: {data_dir.resolve()}")

# Load core tables
df_orders = pd.read_csv(data_dir / "orders.csv")
df_order_items = pd.read_csv(data_dir / "order_items.csv")
df_menu = pd.read_csv(data_dir / "menu_items.csv")
df_categories = pd.read_csv(data_dir / "menu_categories.csv")
df_restaurants = pd.read_csv(data_dir / "restaurants.csv")
df_customers = pd.read_csv(data_dir / "customers.csv")
df_wastage = pd.read_csv(data_dir / "wastage.csv")
df_ratings = pd.read_csv(data_dir / "ratings.csv")

print(f"✓ Orders:        {len(df_orders):,} records")
print(f"✓ Order Items:   {len(df_order_items):,} records")
print(f"✓ Menu Items:    {len(df_menu):,} records")
print(f"✓ Restaurants:   {len(df_restaurants):,} records")
print(f"✓ Customers:     {len(df_customers):,} records")
print(f"✓ Wastage Logs:  {len(df_wastage):,} records")
print(f"✓ Ratings:       {len(df_ratings):,} records")


### 3. Data Hygiene & Quality Verification
Assess data completeness, check for nulls, verify key constraints, and calculate table-level quality health scores.


In [ ]:
quality_audit = []

for name, df in [
    ("Orders", df_orders),
    ("Order Items", df_order_items),
    ("Menu Items", df_menu),
    ("Customers", df_customers),
    ("Wastage", df_wastage),
    ("Ratings", df_ratings)
]:
    total_rows = len(df)
    total_nulls = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()
    completeness_pct = round((1 - (total_nulls / (total_rows * df.shape[1]))) * 100, 2)
    quality_audit.append({
        "Entity": name,
        "Rows": total_rows,
        "Columns": df.shape[1],
        "Missing Cells": total_nulls,
        "Duplicate Rows": duplicate_rows,
        "Data Health %": completeness_pct
    })

audit_df = pd.DataFrame(quality_audit)
print("Data Quality & Hygiene Audit Summary:")
display(audit_df)


### 4. Exploratory Data Analysis (EDA) & Revenue Metrics
Analyze overall business KPIs:
- Total Gross Revenue
- Total Order Volume
- Average Order Value (AOV)
- Revenue distribution by Ordering Channel and Location


In [ ]:
# Join order items with orders and menu items
df_joined = df_order_items.merge(df_orders, on="order_id", how="inner")
df_joined = df_joined.merge(df_menu, on="item_id", how="inner")

total_revenue = df_joined["item_total_price"].sum()
total_orders = df_orders["order_id"].nunique()
aov = total_revenue / total_orders if total_orders > 0 else 0

print(f"=== Executive KPIs ===")
print(f"Total Gross Revenue:   ₨ {total_revenue:,.2f}")
print(f"Total Transactions:    {total_orders:,}")
print(f"Average Order Value:   ₨ {aov:,.2f}")

# Channel Breakdown
channel_summary = df_joined.groupby("order_channel").agg(
    Revenue=("item_total_price", "sum"),
    Orders=("order_id", "nunique"),
    Quantity=("quantity", "sum")
).reset_index()
channel_summary["Revenue Share %"] = (channel_summary["Revenue"] / total_revenue * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Revenue by Channel
axes[0].bar(channel_summary["order_channel"], channel_summary["Revenue"] / 1e5, color=['#f59e0b', '#00a8ff', '#10b981', '#a855f7'], edgecolor='none', width=0.55)
axes[0].set_title("Revenue by Ordering Channel (Lakhs)", fontsize=13, fontweight='bold', pad=10)
axes[0].set_ylabel("Revenue (₨ Lakhs)")
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Revenue Share Donut
axes[1].pie(channel_summary["Revenue Share %"], labels=channel_summary["order_channel"], autopct='%1.1f%%',
            colors=['#f59e0b', '#00a8ff', '#10b981', '#a855f7'], startangle=140,
            wedgeprops=dict(width=0.4, edgecolor='w'))
axes[1].set_title("Revenue Share Distribution", fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.show()


### 5. Multi-Indicator Menu Engineering & Profitability Analysis
Rather than judging items solely by gross sales volume, DineIQ synthesizes:
- **Unit Contribution Margin** = (Sale Price − Cost Price)
- **Volume Sold** & **Total Revenue**
- **Kitchen Wastage Rate %**
- **Repeat Purchase Rate** & **Customer Ratings**
- **Promotional Dependency %**

Classification Quadrants:
1. **Profit Drivers (Stars)**: High contribution margin, high sales volume.
2. **Popular Staples (Plowhorses)**: Low contribution margin, high sales volume.
3. **Hidden Gems (Puzzles)**: High contribution margin, low sales volume.
4. **Underperformers (Dogs)**: Low contribution margin, low sales volume.


In [ ]:
# Compute menu item performance metrics
menu_perf = df_joined.groupby(["item_id", "item_name", "category_id"]).agg(
    total_qty=("quantity", "sum"),
    total_rev=("item_total_price", "sum"),
    base_price=("price", "first"),
    cost_price=("cost_price", "first")
).reset_index()

menu_perf["unit_margin"] = menu_perf["base_price"] - menu_perf["cost_price"]
menu_perf["margin_pct"] = (menu_perf["unit_margin"] / menu_perf["base_price"] * 100).round(2)

# Median thresholds for classification
median_vol = menu_perf["total_qty"].median()
median_margin = menu_perf["unit_margin"].median()

def assign_quadrant(row):
    if row["unit_margin"] >= median_margin and row["total_qty"] >= median_vol:
        return "Profit Driver"
    elif row["unit_margin"] < median_margin and row["total_qty"] >= median_vol:
        return "Popular Staple"
    elif row["unit_margin"] >= median_margin and row["total_qty"] < median_vol:
        return "Hidden Gem"
    else:
        return "Underperformer"

menu_perf["performance_class"] = menu_perf.apply(assign_quadrant, axis=1)

print("Menu Item Classification Distribution:")
display(menu_perf["performance_class"].value_counts())

# Scatter Plot Visualization
fig, ax = plt.subplots(figsize=(10, 6))
palette = {
    "Profit Driver": "#10b981",
    "Popular Staple": "#f59e0b",
    "Hidden Gem": "#38bdf8",
    "Underperformer": "#ef4444"
}

for pclass, color in palette.items():
    subset = menu_perf[menu_perf["performance_class"] == pclass]
    ax.scatter(subset["total_qty"], subset["unit_margin"], label=pclass, color=color, s=70, alpha=0.85, edgecolors='black', linewidth=0.5)

ax.axvline(median_vol, color="gray", linestyle="--", alpha=0.5, label="Median Volume")
ax.axhline(median_margin, color="gray", linestyle=":", alpha=0.5, label="Median Margin")
ax.set_title("Menu Item Performance Classification Quadrant", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("Total Sales Volume (Units)")
ax.set_ylabel("Unit Contribution Margin (₨)")
ax.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


### 6. Customer Intelligence & RFM Segmentation
Calculate Recency, Frequency, and Monetary (RFM) metrics to identify high-value patrons, emerging loyalists, and customers at churn risk.


In [ ]:
df_orders['order_date'] = pd.to_datetime(df_orders['order_timestamp'] if 'order_timestamp' in df_orders.columns else df_orders['order_date'])
max_date = df_orders['order_date'].max()

# Aggregate customer metrics
rfm = df_orders.groupby('customer_id').agg(
    recency=('order_date', lambda d: (max_date - d.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('total_amount', 'sum')
).reset_index()

# Quantile Scoring (1 to 4)
rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first'), q=4, labels=[4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

def assign_segment(row):
    r, f = row['r_score'], row['f_score']
    if r >= 3 and f >= 3:
        return 'Champions / Loyal'
    elif r >= 3 and f < 3:
        return 'Recent / Promising'
    elif r == 2 and f >= 2:
        return 'Needs Attention'
    elif r == 1 and f >= 3:
        return 'At Risk / Churning'
    else:
        return 'Hibernating / Inactive'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

print("Customer RFM Segmentation Breakdown:")
seg_summary = rfm.groupby('segment').agg(
    Customers=('customer_id', 'count'),
    Avg_Recency_Days=('recency', 'mean'),
    Avg_Frequency=('frequency', 'mean'),
    Avg_Spend=('monetary', 'mean')
).reset_index()
display(seg_summary)


### 7. Market Basket Analysis & Cross-Sell Affinity
Mine transaction order items to find frequent dish combinations.
We evaluate pairs using:
- **Support**: $P(A \cap B)$
- **Confidence**: $P(B \mid A)$
- **Lift**: $\frac{P(A \cap B)}{P(A) \cdot P(B)}$ (Lift > 1 indicates strong affinity)


In [ ]:
# Extract dish co-occurrences per order
order_baskets = df_order_items.groupby('order_id')['item_id'].apply(list)
num_baskets = len(order_baskets)

item_counts = {}
pair_counts = {}

for basket in order_baskets:
    unique_items = sorted(list(set(basket)))
    for it in unique_items:
        item_counts[it] = item_counts.get(it, 0) + 1
    for i in range(len(unique_items)):
        for j in range(i + 1, len(unique_items)):
            pair = (unique_items[i], unique_items[j])
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

# Convert item IDs to names
item_id_to_name = dict(zip(df_menu['item_id'], df_menu['item_name']))

rules = []
for (itA, itB), count in sorted(pair_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    supAB = count / num_baskets
    supA = item_counts[itA] / num_baskets
    supB = item_counts[itB] / num_baskets
    confAtoB = supAB / supA if supA > 0 else 0
    lift = supAB / (supA * supB) if (supA * supB) > 0 else 0
    rules.append({
        "Item A": item_id_to_name.get(itA, itA),
        "Item B": item_id_to_name.get(itB, itB),
        "Support %": round(supAB * 100, 2),
        "Confidence %": round(confAtoB * 100, 2),
        "Lift": round(lift, 2)
    })

print("Top 10 Cross-Selling Market Basket Rules:")
display(pd.DataFrame(rules))


### 8. Machine Learning Classification Pipeline
Train and evaluate supervised classification models to predict **Menu Performance Class** based on multidimensional feature inputs:
- Models evaluated: **Random Forest Classifier**, **Logistic Regression**, and **XGBoost Classifier**.
- Metrics calculated: **Accuracy**, **Precision (Macro)**, **Recall (Macro)**, and **Macro F1-Score**.


In [ ]:
# Prepare features and target from classified menu items
features = [
    'total_qty', 'total_rev', 'unit_margin', 'margin_pct'
]

X = menu_perf[features].values
y_raw = menu_perf['performance_class'].values

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_labels = le.classes_

# Split into 80% Train, 20% Test (Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Standardize feature scales
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(max_iter=500, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.08, eval_metric='mlogloss', random_state=42)
}

results = []

for mname, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='macro', zero_division=0)
    rec = recall_score(y_test, preds, average='macro', zero_division=0)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    
    results.append({
        "Model": mname,
        "Accuracy": round(acc * 100, 2),
        "Macro Precision": round(prec * 100, 2),
        "Macro Recall": round(rec * 100, 2),
        "Macro F1-Score": round(f1 * 100, 2)
    })

eval_df = pd.DataFrame(results)
print("=== Model Performance Comparison ===")
display(eval_df)


### 9. Dual-Pipeline Cross-Engine Verification
Verify consistency between independent analytic pipelines.
We check agreement across independent models and inspect borderline instances to ensure robust model generalizability.


In [ ]:
# Compare Random Forest vs XGBoost predictions
rf_preds = models["Random Forest"].predict(X_test_scaled)
xgb_preds = models["XGBoost"].predict(X_test_scaled)

agreement = (rf_preds == xgb_preds).mean() * 100
print(f"Dual-Pipeline Model Agreement Rate: {agreement:.2f}%")

# Confusion Matrix for Primary Model (XGBoost)
cm = confusion_matrix(y_test, xgb_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title("XGBoost Confusion Matrix (Test Set)", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.tight_layout()
plt.show()


### 10. Prescriptive What-If Econometric Simulation
DineIQ incorporates econometric simulation to forecast the revenue and profit margin impact of price adjustments:
$$\Delta Q = Q_0 \cdot \epsilon \cdot \left(\frac{\Delta P}{P_0}\right)$$
where $\epsilon$ represents empirical price elasticity of demand.

*Note: All simulated outputs are designated as planning estimates derived from historical demand structure.*


In [ ]:
def simulate_price_adjustment(current_price, current_qty, cost_price, price_change_pct, elasticity=-1.25):
    pct_decimal = price_change_pct / 100.0
    new_price = current_price * (1.0 + pct_decimal)
    
    # Calculate demand volume response with elasticity
    demand_change_pct = pct_decimal * elasticity
    new_qty = max(0, current_qty * (1.0 + demand_change_pct))
    
    old_rev = current_price * current_qty
    new_rev = new_price * new_qty
    
    old_margin = (current_price - cost_price) * current_qty
    new_margin = (new_price - cost_price) * new_qty
    
    return {
        "Price Change %": f"{price_change_pct:+d}%",
        "New Price (₨)": round(new_price, 2),
        "New Quantity": round(new_qty, 1),
        "Revenue Delta (₨)": round(new_rev - old_rev, 2),
        "Margin Delta (₨)": round(new_margin - old_margin, 2),
        "Outcome": "Positive" if new_margin > old_margin else "Negative"
    }

# Run simulation on a representative dish (e.g. Chicken Biryani)
test_dish = menu_perf.iloc[0]
scenarios = [-15, -10, -5, +5, +10, +15]

simulation_results = [
    simulate_price_adjustment(
        test_dish["base_price"],
        test_dish["total_qty"],
        test_dish["cost_price"],
        pct,
        elasticity=-1.15
    )
    for pct in scenarios
]

print(f"Econometric What-If Simulation for: {test_dish['item_name']} (Base Price: ₨ {test_dish['base_price']}, Cost: ₨ {test_dish['cost_price']})")
display(pd.DataFrame(simulation_results))


### 11. Summary & Key Business Takeaways
1. **Multidimensional Assessment**: Evaluating items beyond gross volume prevents subsidizing unprofitable menu items.
2. **Actionable Customer Segmentation**: Targeting 'At Risk' customers with personalized offers preserves recurring CLV.
3. **Optimized Cross-Selling**: Pairing dishes with high Lift scores systematically expands Average Order Value (AOV).
4. **Independent Model Verification**: Cross-pipeline verification guarantees high fidelity and production stability.
5. **Responsible Econometric Planning**: What-If price simulations support evidence-backed revenue management.
